In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
from sklearn.model_selection import KFold
import gc
import numpy as np
import pandas as pd
import re
import time
from tqdm import tqdm
import pickle
import json
import numpy as np
import Python_Modules.HelperFunctions as HelperFunctions
import ast
from sklearn.feature_extraction.text import CountVectorizer
import random

os.chdir('/media/my_drives')
seed = 1
random.seed(seed)                     # Python random
np.random.seed(seed)                  # NumPy random
torch.manual_seed(seed)               # PyTorch CPU random
torch.cuda.manual_seed(seed)          # PyTorch GPU random
torch.cuda.manual_seed_all(seed)      # If using multiple GPUs
torch.backends.cudnn.deterministic = True  # Force cuDNN to be deterministic
torch.backends.cudnn.benchmark = False     # Disable cuDNN auto-tuner (more reproducible)
os.environ['PYTHONHASHSEED'] = str(seed)

# Convert the prob dicts into a matrix
def dicts_to_matrix(prob_dicts, classes):
    """
    Converts a list of probability dictionaries into a 2D numpy array.
    Each row corresponds to one sample, columns correspond to classes.
    """
    matrix = []
    for d in prob_dicts:
        row = [d.get(c, 0.0) for c in classes]  # fallback to 0 if class not in dict
        matrix.append(row)
    return np.array(matrix)

In [2]:
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim, n_classes, dropout):
        super(SimpleClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.dropout = nn.Dropout(dropout)
        if n_classes > 1:
            self.fc2 = nn.Linear(256, n_classes)
            self.final_activation = nn.LogSoftmax(dim=1)  # for CrossEntropyLoss, no activation needed, but can still use LogSoftmax
        else:
            self.fc2 = nn.Linear(256, 1)
            self.final_activation = nn.Sigmoid()

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        if self.final_activation is not None:
            x = self.final_activation(x)
        return x

# Early stopping helper
class EarlyStopping:
    def __init__(self, patience=10):
        self.patience = patience
        self.counter = 0
        self.best_loss = None
        self.best_model_state = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_model_state = model.state_dict()
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

# Training function
def train_model(model, optimizer, criterion, train_loader, val_loader, max_epochs=100, patience=10, device='cpu'):
    early_stopping = EarlyStopping(patience)
    history = []
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            output = model(X_batch)

            if output.shape[1] > 1:  # multi-class
                loss = criterion(output, y_batch)
            else:  # binary
                loss = criterion(output.squeeze(), y_batch.float())

            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                output = model(X_batch)

                if output.shape[1] > 1:
                    loss = criterion(output, y_batch)
                    preds = torch.argmax(output, dim=1)
                else:
                    loss = criterion(output.squeeze(), y_batch.float())
                    preds = (output.squeeze() > 0.5).long()

                val_loss += loss.item()
                correct += (preds == y_batch).sum().item()
                total += y_batch.size(0)

        val_loss /= len(val_loader)
        val_acc = correct / total
        history.append((val_loss, val_acc))

        early_stopping(val_loss, model)

        if early_stopping.early_stop:
            model.load_state_dict(early_stopping.best_model_state)
            break

    return model, history

In [171]:
DF_PATH = 'DATA4/max/Image_Benchmark/R1_Analytics/Results.csv'
df = pd.read_csv(DF_PATH)
df.tail(2)

,Model,Dataset,N_Training_Samples,BATCH_SIZE,LEARNING_RATE,EPOCHS,Time_Seconds,Accuracy,BIGRAM_BOOL,BINARY_BOOL
3769,Ensemble_AllProba,FashionImages,1000.0,64.0,0.00100,32.0,0.63,0.921569,NaN,NaN
3770,Ensemble_AllProba,FashionImages,1000.0,64.0,0.00001,32.0,0.63,0.422969,NaN,NaN


In [ ]:
MODEL_NAME = 'Ensemble_AllProba'
datasets = [
 'AIDA',
 'Brand-Styles',
 'BrandSelfie',
 'E-Commerce',
 'Emotion-6',
 'FILGRIM',
 'FashionImages',
 'GenImgNet',
 'Generated-Ethnicity',
 'Generated-Gender',
 'KonIQ',
 'LogoDet-3K',
 'StoreItemColor',
 'UnBiasedEmo',
 'Unsplash-Images',
 'good-bad',
 'intel-sceneries',
 'sentiment']

EPOCHS = 32
BATCH_SIZES = [16, 32, 64]
LEARNING_RATES = [0.01, 0.001, 1e-5]

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for DATASET in tqdm(datasets):
    for N_Samples in [1000, 200]:
        if (DATASET == 'AIDA') & (N_Samples == 1000):
            continue
            
        print("Run", DATASET, N_Samples)
        for BATCH_SIZE in BATCH_SIZES:
            for LEARNING_RATE in LEARNING_RATES:
                if df.loc[((df.Model == MODEL_NAME) & (df.Dataset == DATASET) & (df.N_Training_Samples == N_Samples) & (df.BATCH_SIZE == BATCH_SIZE) & (df.LEARNING_RATE == LEARNING_RATE))].shape[0] == 0:
                    try:
                        DATASET_CSV = pd.read_csv(f"DATA4/max/Image_Benchmark/R1_Analytics/CSV_Training_{N_Samples}/{DATASET}.csv")
                        DATASET_CSV = DATASET_CSV.dropna(subset=['prediction_gpt5_allclassprobs', 'prediction_convnext_allclassprobs'])
        
                        DATASET_CSV_HOLDOUT = pd.read_csv(f"DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET}.csv")
                        train_labels = set(DATASET_CSV['label']).intersection(set(DATASET_CSV_HOLDOUT['label']))
                        DATASET_CSV = DATASET_CSV[DATASET_CSV['label'].isin(train_labels)]
                        DATASET_CSV_HOLDOUT = DATASET_CSV_HOLDOUT[DATASET_CSV_HOLDOUT['label'].isin(train_labels)]

                        label_to_int = {label: str(idx) for idx, label in enumerate(sorted(train_labels))}
                        DATASET_CSV['label_num'] = DATASET_CSV['label'].map(label_to_int)
                        DATASET_CSV_HOLDOUT['label_num'] = DATASET_CSV_HOLDOUT['label'].map(label_to_int)                            

                        classes = DATASET_CSV.label.unique()

                        # Extract the list of prob dicts from your dataframe
                        gpt_dicts = DATASET_CSV['prediction_gpt5_allclassprobs'].apply(json.loads).tolist()
                        convnext_dicts = DATASET_CSV['prediction_convnext_allclassprobs'].apply(json.loads).tolist()

                        # Remap convnext_dicts to use label names instead of stringified ints
                        int_to_label = {v: k for k, v in label_to_int.items()}
                        convnext_dicts = [
                            {label: d.get(k, 0.0) for k, label in int_to_label.items()}
                            for d in convnext_dicts
                        ]
                                                        
                        # Convert dicts to matrices
                        X_gpt = dicts_to_matrix(gpt_dicts, classes)
                        X_convnext = dicts_to_matrix(convnext_dicts, classes)
                        
                        # Concatenate GPT + ConvNeXt probabilities
                        X = np.hstack([X_gpt, X_convnext])
                        
                        # Labels
                        y = pd.Categorical(DATASET_CSV['label_num']).codes
                        

                        input_dim = X.shape[1]
                        n_classes = len(np.unique(y))

                        X_train = torch.tensor(X, dtype=torch.float32)
                        y_train = torch.tensor(y, dtype=torch.long)

                        # Extract the list of prob dicts from your dataframe
                        gpt_dicts_val = DATASET_CSV_HOLDOUT['prediction_gpt5_allclassprobs'].apply(json.loads).tolist()
                        convnext_dicts_val = DATASET_CSV_HOLDOUT[f'prediction_convnext_allclassprobs_{N_Samples}'].apply(json.loads).tolist()

                        # Remap convnext_dicts to use label names instead of stringified ints
                        convnext_dicts_val = [
                            {label: d.get(k, 0.0) for k, label in int_to_label.items()}
                            for d in convnext_dicts_val
                        ]
                        
                        # Convert dicts to matrices
                        X_gpt_val = dicts_to_matrix(gpt_dicts_val, classes)
                        X_convnext_val = dicts_to_matrix(convnext_dicts_val, classes)
                        
                        # Concatenate GPT + ConvNeXt probabilities
                        X_val = np.hstack([X_gpt_val, X_convnext_val])
                        
                        # Labels
                        y_val = pd.Categorical(DATASET_CSV_HOLDOUT['label_num']).codes

                        X_val = torch.tensor(X_val, dtype=torch.float32)
                        y_val = torch.tensor(y_val, dtype=torch.long)

                        train_ds = TensorDataset(X_train, y_train)
                        val_ds = TensorDataset(X_val, y_val)

                        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
                        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
                    
                        model = SimpleClassifier(input_dim, n_classes, dropout=0.25).to(device)

                        if n_classes > 1:
                            criterion = nn.CrossEntropyLoss()
                        else:
                            criterion = nn.BCELoss()

                        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

                        start_time = time.time()
                        model, history = train_model(model, optimizer, criterion, train_loader, val_loader, max_epochs=EPOCHS, patience=10, device=device)
                        total_time = np.round(time.time() - start_time, 2)

                        best_idx = min(range(len(history)), key=lambda i: history[i][0])
                        best_epoch = best_idx + 1
                        best_val_loss = history[best_idx][0]
                        best_val_acc = history[best_idx][1]

                        index_row = df.shape[0]
                        df.at[index_row, 'Model'] = MODEL_NAME
                        df.at[index_row, 'Dataset'] = DATASET
                        df.at[index_row, 'N_Training_Samples'] = N_Samples
                        df.at[index_row, 'BATCH_SIZE'] = BATCH_SIZE
                        df.at[index_row, 'LEARNING_RATE'] = LEARNING_RATE
                        df.at[index_row, 'EPOCHS'] = best_epoch
                        df.at[index_row, 'Time_Seconds'] = total_time
                        df.at[index_row, 'Accuracy'] = best_val_acc

                        df.to_csv(DF_PATH, index=False)
                        del DATASET_CSV
                        del X, y
                        del model
                        gc.collect()
                    except Exception as e:
                            print('Error:', e)
                            pass

  0%|                                                     | 0/1 [00:00<?, ?it/s]

Run FashionImages 1000
Run FashionImages 200


100%|█████████████████████████████████████████████| 1/1 [00:05<00:00,  5.69s/it]
